In [4]:
from build123d import *
from build123d import Rotation

from build123d import *
# -----------------------------
# 盘子（固定）
# -----------------------------
class BearingDisk(Compound):
    def __init__(
        self,
        diameter: float,
        thickness: float,
        bearing_diameter: float,
        bearing_depth: float,
        shaft_diameter: float,
    ):
        with BuildPart() as disk_part:
            # 主体圆盘
            with BuildSketch():
                Circle(diameter / 2)
            extrude(amount=thickness)

            # 轴承座（浅孔）
            with BuildSketch(Plane.XY):
                Circle(bearing_diameter / 2)
            extrude(amount=-bearing_depth, mode=Mode.SUBTRACT)

            # 中轴孔（贯穿）
            with BuildSketch(Plane.XY):
                Circle(shaft_diameter / 2)
            extrude(amount=thickness, mode=Mode.SUBTRACT)

            # ⚡ 父 joint：固定
            RigidJoint(
                label="shaft_joint",
                joint_location=Location((0, 0, thickness / 2))
            )

        super().__init__(
            disk_part.part.wrapped,
            joints=disk_part.part.joints
        )

# -----------------------------
# 轴承（可旋转）
# -----------------------------
class CylinderBearing(Compound):
    def __init__(
        self,
        outer_diameter: float,
        height: float,
        inner=True
    ):
        with BuildPart() as bearing_part:
            # 外圈圆柱
            with BuildSketch(Plane.XY):
                Circle(outer_diameter / 2)
            extrude(amount=height)

            # ⚡ child joint：可旋转
            RevoluteJoint(
                label="bearing_axis",
                axis=Axis((0, 0, height / 2), (0, 0, 1)),
                angular_range=(0, 360)
            )
            if inner: 
                RigidJoint( 
                    "hinge_axis", joint_location=Location( (0,0, 0) ),
                ) 
            else: 
                RevoluteJoint( "hinge_axis", axis=Axis( (0,0, 0), (1, 0,0) ), angular_range=(90, 270), )

        super().__init__(
            bearing_part.part.wrapped,
            joints=bearing_part.part.joints
        )

In [ ]:
# 创建零件
d1 = BearingDisk(
    diameter=50,
    thickness=1,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2
)
d2 = BearingDisk(
    diameter=50,
    thickness=1,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2
)
b1 = CylinderBearing(
    outer_diameter=2,
    # inner_diameter=8.2,
    height=10,
    inner=True
)
b2 = CylinderBearing(
    outer_diameter=1,
    # inner_diameter=8.2,
    height=20,
    inner=False
)


b2.joints["hinge_axis"].connect_to(b1.joints["hinge_axis"], angle=110)
# b2.locate(Location((0,0,0), (90,0,0))) 
# 创建装配
# asm = Assembly()
# asm.add(bearing)
# asm.add(disk)

# 对齐盘子和轴承
# disk.part.joints["shaft_joint"].connect_to(bearing.part.joints["bearing_axis"])
b1.joints["bearing_axis"].connect_to(d1.joints["shaft_joint"], angle=50)
b2.joints["bearing_axis"].connect_to(d2.joints["shaft_joint"])

In [ ]:
from build123d import (
     Compound
)
assembly = Compound(label="assembly", children=[
    # box, lid, hinge_inner, hinge_outer
    # d1,b1,d2,b2
    # d1,d2
    b1,b2,d1,d2
])

In [5]:
from build123d import *

class Gimbal(Compound):
    def __init__(
        self, 
        disc_diameter_gap: float,
        disc_thickness_gap: float,
        frame_thickness_diameter: float, 
        corner_radius: float,
        connector_length: float,
        connector_radius: float
    ):
        """
        创建一个用于支撑转盘的 Gimbal 框架
        disc_diameter_gap: 转盘直径 + 留出间隙
        disc_thickness_gap: 转盘厚度 + 留出间隙
        frame_thickness_diameter: 矩形管管径
        corner_radius: 方孔圆角半径
        connector_length: 左右连接柱长度
        connector_radius: 左右连接柱半径
        """
        a = disc_diameter_gap + frame_thickness_diameter/2
        b = disc_thickness_gap + frame_thickness_diameter/2
        r = corner_radius
        tube_radius = connector_radius
        tube_length = connector_length

        with BuildPart() as gimbal_part:
            # -----------------------------
            # 1️⃣ 构建圆角矩形路径
            with BuildLine() as frame:
                start = (-a/2 + r, -b/2)
                Line(start, (a/2 - r, -b/2))
                JernArc(start=(a/2 - r, -b/2), tangent=(1, 0), radius=r, arc_size=90)
                Line((a/2, -b/2 + r), (a/2, b/2 - r))
                JernArc(start=(a/2, b/2 - r), tangent=(0, 1), radius=r, arc_size=90)
                Line((a/2 - r, b/2), (-a/2 + r, b/2))
                JernArc(start=(-a/2 + r, b/2), tangent=(-1, 0), radius=r, arc_size=90)
                Line((-a/2, b/2 - r), (-a/2, -b/2 + r))
                JernArc(start=(-a/2, -b/2 + r), tangent=(0, -1), radius=r, arc_size=90)
            rectangle_wire = frame.wire()

            # -----------------------------
            # 2️⃣ sweep 圆角矩形管
            # TODO: start and tangent
            start = rectangle_wire @ 0
            tangent = rectangle_wire % 0 
            with BuildSketch(Plane(origin=start, z_dir=tangent)) as sk:
                Circle(frame_thickness_diameter/2)
            sweep(sk.sketch, rectangle_wire, is_frenet=True)

            # -----------------------------
            # 3️⃣ 左右中点圆柱（extrude）沿 X 方向
            left_mid = (-a/2, 0)
            right_mid = (a/2, 0)
        
            for mid, direction in [(left_mid, -1), (right_mid, 1)]:
                plane = Plane(origin=(mid[0], mid[1], 0), z_dir=(direction, 0, 0))
                with BuildSketch(plane) as skc:
                    Circle(connector_radius)
                extrude(skc.sketch, connector_length)
                
            # constraints 
            RevoluteJoint(
                label="bearing_axis",
                axis=Axis((0, 0, 0), (0, 1, 0)),
                angular_range=(0, 360)
            )
            
        super().__init__(
            gimbal_part.part.wrapped,
            joints=gimbal_part.part.joints
        )



In [6]:
g = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2
)
d = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2
)
g.joints["bearing_axis"].connect_to(d.joints["shaft_joint"], angle=50)
# g

In [7]:
gimbal =Compound(label="gim", children=[
    # box, lid, hinge_inner, hinge_outer
    # d1,b1,d2,b2
    # d1,d2
    g,d
]) 

In [8]:
gimbal

Compound at 0x7f5a3831d3d0, label(gim), #children(2)